# Basic RNN

In [1]:
import torch 
import torch.nn as nn
torch.manual_seed(1)

rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

w_xh = rnn_layer.weight_ih_l0      
w_hh = rnn_layer.weight_hh_l0    
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0

print('W_xh shape:', w_xh.shape)
print('W_hh shape:', w_hh.shape)
print('b_xh shape:', b_xh.shape)
print('b_hh shape:', b_hh.shape)

x_seq = torch.tensor([[1.0]*5, [2.0]*5, [3.0]*5]).float()

output, hn = rnn_layer(torch.reshape(x_seq, (1, 3, 5)))

out_man = []
h_prev = torch.zeros((1, w_hh.shape[0]))  

for t in range(3):
    xt = x_seq[t].unsqueeze(0)  

    preact = xt @ w_xh.T + b_xh + h_prev @ w_hh.T + b_hh
    ht = torch.tanh(preact)

    out_man.append(ht)
    h_prev = ht

    print(f'Time step {t} =>')
    print('   Input           :', xt.numpy())
    print('   Pre-activation  :', preact.detach().numpy())
    print('   Output (manual) :', ht.detach().numpy())
    print('   RNN output      :', output[:, t].detach().numpy())
    print()

W_xh shape: torch.Size([2, 5])
W_hh shape: torch.Size([2, 2])
b_xh shape: torch.Size([2])
b_hh shape: torch.Size([2])
Time step 0 =>
   Input           : [[1. 1. 1. 1. 1.]]
   Pre-activation  : [[-0.36770207  0.5835656 ]]
   Output (manual) : [[-0.3519801   0.52525216]]
   RNN output      : [[-0.3519801   0.52525216]]

Time step 1 =>
   Input           : [[2. 2. 2. 2. 2.]]
   Pre-activation  : [[-0.8370501  0.9979756]]
   Output (manual) : [[-0.68424344  0.76074266]]
   RNN output      : [[-0.68424344  0.76074266]]

Time step 2 =>
   Input           : [[3. 3. 3. 3. 3.]]
   Pre-activation  : [[-1.3126388  1.4973243]]
   Output (manual) : [[-0.8649416   0.90466356]]
   RNN output      : [[-0.8649416   0.90466356]]



# Implementing RNNs for sequence modeling in PyTorch

##  Project one – predicting the sentiment of IMDb movie reviews

In [1]:
from torchtext.datasets import IMDB
train_dataset = IMDB(split='train')
test_dataset = IMDB(split='test')

c:\Users\Aksha\anaconda3\envs\torch-env\lib\site-packages\torchtext\datasets\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\Aksha\anaconda3\envs\torch-env\lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)


ImportError: cannot import name 'DILL_AVAILABLE' from 'torch.utils.data.datapipes.utils.common' (c:\Users\Aksha\anaconda3\envs\torch-env\lib\site-packages\torch\utils\data\datapipes\utils\common.py)

In [ ]:


## Step 1: create the datasets
from torch.utils.data.dataset import random_split
torch.manual_seed(1)
train_dataset, valid_dataset = random_split(list(train_dataset), [20000, 5000])

## Step 2: find unique tokens (words)
import re
from collections import Counter, OrderedDict
def tokenizer(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
    text = re.sub('[\W]+', ' ', text.lower()) +' '.join(emoticons).replace('-', '')
    tokenized = text.split()
    return tokenized

token_counts = Counter()
for label, line in train_dataset:
    tokens = tokenizer(line)
    token_counts.update(tokens)
print('Vocab-size:', len(token_counts))


Vocab-size: 69023


In [12]:
from collections import OrderedDict
from torchtext.vocab import Vocab

sorted_by_freq_tuples = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
ordered_dict = OrderedDict(sorted_by_freq_tuples)

vocab = Vocab(ordered_dict)
vocab.insert_token("<pad>", 0)
vocab.insert_token("<unk>", 1)
vocab.set_default_index(vocab["<unk>"])

OSError: [WinError 127] The specified procedure could not be found